# Wild Drone LLM Workshop: Part 2 - Drone Safari Command Agent

Welcome to Part 2! Now that you understand how LLMs work and how tools give them superpowers, let's apply this to a fun game scenario.

## What You'll Build

A smart drone command agent that:
- Understands natural language instructions
- Translates them into precise game actions
- Navigates a safari environment safely
- Helps you photograph wildlife

## The Drone Safari Game

You control a drone in a 12x12 grid safari park. Your mission:
- **Navigate safely** - Avoid trees and boundaries
- **Find animals** - Zebras, elephants, and oryx roam the park  
- **Take photos** - You have 5 pictures to capture all 3 species
- **Stay alert** - Get too close and animals will run away!

## Learning Journey

1. **Explore the Game** - Understand the safari environment
2. **Build Basic Agent** - Create a drone that follows simple commands
3. **Add Tools** - Give your agent game actions
4. **Test Commands** - See how natural language becomes game moves
5. **Experiment** - Try different prompts and strategies

Let's start exploring! 🚁🦓🐘

## Install Required Packages and Download Scripts
(Just run these cells, no need to understand them)

---

In [ ]:
# Install required packages
!pip install litellm matplotlib numpy python-dotenv
# Download the scripts and images necessary for the notebooks
!wget https://raw.githubusercontent.com/alejp1998/wilddrone-llm-workshop/main/drone_safari_game.py -O drone_safari_game.py
!wget https://raw.githubusercontent.com/alejp1998/wilddrone-llm-workshop/main/llm_agents.py -O llm_agents.py
!mkdir -p images

!wget https://raw.githubusercontent.com/alejp1998/wilddrone-llm-workshop/main/images/tree.png -O images/tree.png
!wget https://raw.githubusercontent.com/alejp1998/wilddrone-llm-workshop/main/images/zebra.png -O images/zebra.png
!wget https://raw.githubusercontent.com/alejp1998/wilddrone-llm-workshop/main/images/elephant.png -O images/elephant.png
!wget https://raw.githubusercontent.com/alejp1998/wilddrone-llm-workshop/main/images/oryx.png -O images/oryx.png
!wget https://raw.githubusercontent.com/alejp1998/wilddrone-llm-workshop/main/images/drone.png -O images/drone.png
!wget https://raw.githubusercontent.com/alejp1998/wilddrone-llm-workshop/main/images/crashed_drone.png -O images/crashed_drone.png
!wget https://raw.githubusercontent.com/alejp1998/wilddrone-llm-workshop/main/images/scared_animal.png -O images/scared_animal.png
!wget https://raw.githubusercontent.com/alejp1998/wilddrone-llm-workshop/main/images/shining.png -O images/shining.png

## Setup: Define interactive interface classes for playing and testing our implementations
(Just run these cells again, no need to understand them)

---

In [ ]:
# Game Interface
# This creates a simple button interface to help you understand the game mechanics

import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
from matplotlib.patches import Rectangle
import threading
import time

class GameInterface:
    """Simple button-controlled interface for the drone safari game"""

    def __init__(self, game):
        """Initialize with a game instance"""
        self.game = game
        self.game.__init__()  # Reset game state
        self.output_area = widgets.Output()
        self.result_area = widgets.Output()

    def create_interface(self):
        """Create the button-controlled interface"""
        # Create control buttons
        move_forward_btn = widgets.Button(description="↑ Forward", button_style='info', layout=widgets.Layout(width='90px'))
        move_left_btn = widgets.Button(description="← Left", button_style='info', layout=widgets.Layout(width='90px'))
        move_right_btn = widgets.Button(description="→ Right", button_style='info', layout=widgets.Layout(width='90px'))
        move_backward_btn = widgets.Button(description="↓ Backward", button_style='info', layout=widgets.Layout(width='90px'))

        turn_left_btn = widgets.Button(description="⟲ Turn L", button_style='warning', layout=widgets.Layout(width='90px'))
        turn_right_btn = widgets.Button(description="⟳ Turn R", button_style='warning', layout=widgets.Layout(width='90px'))

        take_picture_btn = widgets.Button(description="📷 Photo", button_style='', layout=widgets.Layout(width='90px'))
        restart_btn = widgets.Button(description="🔄 Restart", button_style='danger', layout=widgets.Layout(width='90px'))

        # Set up button handlers
        move_forward_btn.on_click(lambda b: self._handle_action('move', 'forward'))
        move_left_btn.on_click(lambda b: self._handle_action('move', 'left'))
        move_right_btn.on_click(lambda b: self._handle_action('move', 'right'))
        move_backward_btn.on_click(lambda b: self._handle_action('move', 'backward'))
        turn_left_btn.on_click(lambda b: self._handle_action('turn', 'left'))
        turn_right_btn.on_click(lambda b: self._handle_action('turn', 'right'))
        take_picture_btn.on_click(lambda b: self._handle_action('picture', ''))
        restart_btn.on_click(lambda b: self._handle_action('restart', ''))

        # Create layout - reorganized as requested
        row1 = widgets.HBox([turn_left_btn, move_forward_btn, turn_right_btn])
        row2 = widgets.HBox([move_left_btn, take_picture_btn, move_right_btn])
        row3 = widgets.HBox([restart_btn, move_backward_btn, widgets.HTML(value="<div style='width:90px'></div>")])

        button_controls = widgets.VBox([
            row1, row2, row3
        ])

        # Create the main interface
        interface = widgets.VBox([
            self.output_area,
            self.result_area,
            button_controls
        ])

        # Show initial state
        with self.output_area:
            self._show_game_state()

        with self.result_area:
            print("🎮 Ready to play! Movement is relative to where the drone is facing.")

        display(interface)

    def _handle_action(self, action_type, direction):
        """Handle game actions"""
        try:
            if action_type == 'move':
                result = self.game.move(direction)
            elif action_type == 'turn':
                result = self.game.turn(direction)
            elif action_type == 'picture':
                result = self.game.take_picture()
            elif action_type == 'restart':
                self.game.__init__()
                result = "🔄 Game restarted!"
            else:
                result = "Unknown action"

            # Update display
            with self.output_area:
                clear_output(wait=True)
                self._show_game_state()

            with self.result_area:
                clear_output(wait=True)
                print(f"🎮 {result}")

        except Exception as e:
            with self.result_area:
                clear_output(wait=True)
                print(f"❌ Error: {e}")

    def _show_game_state(self):
        """Display the current game state"""
        # Clear matplotlib state
        plt.ioff()
        plt.close('all')

        # Create visualization
        fig = self.game.visualize(figsize=(8, 6))

        # Show and close
        plt.show()
        plt.close(fig)

print("✅ Button Game Interface loaded successfully!")

In [ ]:
# NL Game Interface
# This creates an interactive interface for testing your command agent

import matplotlib.pyplot as plt
import matplotlib
# Ensure we're using the right backend for Jupyter
matplotlib.use('module://matplotlib_inline.backend_inline')

class NLGameInterface:
    """Interactive widget for Jupyter notebooks to test command agents"""

    def __init__(self, title, game, agent):
        """Initialize with a game instance and an agent"""
        self.title = title
        self.game = game
        self.agent = agent
        self.history = []
        self.interface_created = False

        # Call the init method on the game to ensure it's properly set up
        self.game.__init__()

        try:
            from IPython.display import display, clear_output
            import ipywidgets as widgets
            self.display = display
            self.clear_output = clear_output
            self.widgets = widgets
            self.ipython_available = True
        except ImportError:
            print("Warning: IPython widgets not available. Please install jupyter widgets:")
            print("pip install ipywidgets")
            self.ipython_available = False

    def create_interface(self):
        """Create the interactive interface"""
        if not self.ipython_available:
            print("Interactive interface not available. Please install ipywidgets.")
            return

        # Prevent creating multiple interfaces
        if self.interface_created:
            print("🔄 Interface already created! Use the existing interface above.")
            return

        # Create widgets
        self.command_input = self.widgets.Text(
            placeholder="Enter command (e.g., 'move forward', 'turn left', 'take picture')",
            description="Command:",
            style={'description_width': 'initial'},
            layout=self.widgets.Layout(width='450px'),
            continuous_update=False  # This prevents constant updates
        )

        self.send_button = self.widgets.Button(
            description="Send Command",
            button_style='primary',
            layout=self.widgets.Layout(width='120px')
        )

        self.restart_button = self.widgets.Button(
            description="Restart",
            button_style='warning',
            layout=self.widgets.Layout(width='120px')
        )

        self.output_area = self.widgets.Output()

        # Set up event handlers - using the modern approach
        self.command_input.observe(self._on_command_change, names='value')
        self.send_button.on_click(self._on_send_click)
        self.restart_button.on_click(self._on_restart_click)

        # Create layout
        input_box = self.widgets.HBox([self.command_input, self.send_button, self.restart_button])
        interface = self.widgets.VBox([
            self.widgets.HTML(f"<h3>🚁 {self.title}</h3>"),
            input_box,
            self.output_area
        ])

        # Display initial state
        with self.output_area:
            self._show_initial_state()

        self.display(interface)
        self.interface_created = True
        # DO NOT RETURN INTERFACE TO PREVENT DUPLICATE DISPLAYS
        # return interface

    def _on_command_change(self, change):
        """Handle Enter key press in command input (modern approach)"""
        # Only process if this was triggered by Enter key (when value actually changes)
        if change['type'] == 'change' and change['name'] == 'value':
            self._process_command()

    def _on_send_click(self, button):
        """Handle send button click"""
        self._process_command()

    def _on_restart_click(self, button):
        """Handle restart button click"""
        # Reset the game to initial state
        self.game.__init__()  # Reinitialize the game
        self.history = []  # Clear command history

        # Clear and refresh the display
        with self.output_area:
            self.clear_output(wait=True)
            print("🔄 Game Restarted!")
            print("="*50)
            self._show_initial_state()

    def _process_command(self):
        """Process the command and update display"""
        command = self.command_input.value.strip()
        if not command:
            return

        try:
            # Get agent response with debug information
            if hasattr(self.agent, 'chat_with_debug'):
                # Use the new debug method if available
                response, debug_output = self.agent.chat_with_debug(command, debug=True, clear_memory=False)
            else:
                # Fallback to regular chat method
                response = self.agent.chat(command)
                debug_output = []

            # Clear output as we received the response
            with self.output_area:
                self.clear_output(wait=True)

            # Get game status
            status = self.game.get_status()

            # Store in history
            self.history.append({
                'command': command,
                'response': response,
                'debug_output': debug_output,
                'status': status
            })

            # Update display
            with self.output_area:
                self.clear_output(wait=True)
                self._show_command_result(command, response, debug_output, status)

        except Exception as e:
            with self.output_area:
                self.clear_output(wait=True)
                print(f"❌ Error processing command: {e}")
                print("Please try again with a different command.")
                import traceback
                print("Debug traceback:")
                print(traceback.format_exc())

        # Clear input for next command
        self.command_input.value = ""

    def _show_initial_state(self):
        """Show the initial game state"""

        # Completely clear matplotlib state
        plt.ioff()  # Turn off interactive mode
        plt.close('all')  # Close all figures

        # Create the visualization
        fig = self.game.visualize(figsize=(6, 6))

        # Explicitly show and then close the figure to prevent accumulation
        plt.show()
        plt.close(fig)

    def _show_command_result(self, command, response, debug_output, status):
        """Show the result of a command with debug information"""

        # Show debug information if available
        if debug_output:
            print("🔍 Agent Debug Information:")
            for debug_line in debug_output:
                print(debug_line)

        # Completely clear matplotlib state
        plt.ioff()  # Turn off interactive mode
        plt.close('all')  # Close all figures

        # Create the visualization
        fig = self.game.visualize(figsize=(6, 6))

        # Explicitly show and then close the figure to prevent accumulation
        plt.show()
        plt.close(fig)

print("✅ Natural-language Game Interface loaded successfully!")

## Setup: Import Required Modules

You need the same **Google API key** from Part 1. If you completed Part 1, you should be all set!

In [ ]:
# Configuration and Setup
import os
from litellm import completion, _turn_on_debug
# _turn_on_debug()
from dotenv import load_dotenv
from llm_agents import add_parameters_schema, LLMAgent
from drone_safari_game import DroneSafariGame
from google.colab import userdata

# Load environment variables from .env file (if it exists)
load_dotenv()

# MODEL CONFIGURATION
# Check if GOOGLE API KEY was correctly configured
if userdata.get('GOOGLE_API_KEY'):
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    # Hide key partially before printing it:
    print("Google API Key ready ✓")
else:
    print("Google API Key was not set succesfully.")

# Set the model to use throughout the workshop
model_name = "gemini/gemini-2.5-flash"  # Using stable model

# Understanding the Drone Safari Game

## First: Play the Game Yourself! 🎮

Before building an AI agent, let's experience the game directly. Below you'll find a **game interface** that lets you play the drone safari game using clickable buttons.

This hands-on experience will help you understand:
- How the drone moves and turns
- The challenge of photographing animals from exactly 2 cells away
- The consequences of crashes and getting too close to animals
- The strategy needed to complete the mission

## Then: Explore the Game Programmatically

After playing, we'll explore the game through code to understand what your AI agent needs to accomplish!

In [ ]:
# 🎮 Try the Game with Button Controls!
# This gives you a hands-on experience with the game mechanics

# Create a new game instance for button control
button_game = DroneSafariGame()

# Create the button interface
button_interface = GameInterface(button_game)
button_interface.create_interface()

In [ ]:
# Create a safari game and explore it
game = DroneSafariGame()

# This is how you programatically interact with the game
# Each action returns a string describing the result of the action so that the agent can understand what happened

# Moving
result = game.move('forward') # either 'forward', 'backward', 'left', 'right'
print(f"Action 1: {result}")

# Turning
result = game.turn('right') # either 'left' or 'right'
print(f"Action 2: {result}")

# Take picture
result = game.take_picture()
print(f"Action 3: {result}")

# Visualize the game state, you can see a trace of the past actions in the visualization
print("\n🗺️ Safari Map:")
game.visualize(figsize=(8, 6))

## Building Your Natural Language Command Agent

Making games accessible helps include everyone, regardless of disability or skill level.  
Traditional controllers and keyboard shortcuts can be hard for people with motor, visual, or cognitive challenges.  
Using simple, natural language commands makes gameplay easier and more welcoming.

**Example commands:**
- "Move forward"
- "Turn left"
- "Take a picture"

Natural language control allows players to enjoy games without memorizing buttons or needing precise movements.  
This approach lowers barriers and supports a more inclusive gaming experience for all.



## Step 1: Define the tools the agent has access to
To interact with the game, we need tools for:
- Moving between cells
- Turning to face different directions
- Taking a picture

In [ ]:
# Tool 1: Drone Movement
@add_parameters_schema(
    move_type={
        "type": "string",
        "description": "Movement type: 'forward', 'backward', 'left', 'right'"
    }
)
def drone_move(move_type: str) -> str:
    """Move the drone in the specified move_type: either forward, backward, left or right"""
    result = game.move(move_type)
    return result

# Test the tool
print("🚁 Testing drone movement:")
print(drone_move('forward'))

In [ ]:
# Tool 2: Drone Turning
@add_parameters_schema(
    turn_direction={
        "type": "string",
        "description": "Turn direction: 'left' or 'right'"
    }
)
def drone_turn(turn_direction: str) -> str:
    """Turn the drone left or right"""
    result = game.turn(turn_direction)

    return result

# Test the tool
print("🔄 Testing drone turning:")
print(drone_turn('right'))

In [ ]:
# Tool 3: Taking Pictures
@add_parameters_schema(
    # This tool doesnt take any arguments, no description is needed
)
def drone_take_picture() -> str:
    """Take a picture with the drone camera"""
    result = game.take_picture()

    return result

# Test the tool
print("📷 Testing drone camera:")
print(drone_take_picture())

## Step 2: Create Your Natural Command Agent

Now we'll create an agent that translates your natural language commands into precise game actions. The key insight:

**🧠 YOU make the strategy decisions**  
**🤖 The AGENT translates your words into actions**

Your agent should:
1. **Listen** to natural language commands
2. **Choose** the right tool (move, turn, or take picture)  
3. **Execute** exactly ONE action
4. **Report** what it did

> **🎯 Keep it Simple**: The agent doesn't need to know game strategy - that's your job! It just needs to understand commands like "move forward", "turn left", "take a picture".

In [ ]:
# TODO: STUDENT TASK - Create your Natural Command Agent

# Reset the game for a fresh start
game = DroneSafariGame()

# 1. Create a new agent (we'll reuse LLMAgent class)
drone_command_agent = LLMAgent(model=model_name)

# 2. Add your tools to the agent (uncomment and complete)
# drone_command_agent.tools = [_____, _____, _____]

# 3. Write your system prompt here
drone_command_agent.system_prompt = """
Write your system prompt here!

Consider including:
- What the agent's role is (command translator, not strategist)
- Available actions: move, turn, take_picture
- That it should execute ONE action per command
- How to turn into actions natural-language commands like "move forward", "turn left", "take picture"
- Ask it to provide a summary of the action taken and its consequences in the game.
"""

## 🧪 Test Your Natural Command Agent - Interactive Interface

**Basic Commands to try:**
- *"Move forward"* → Should use `drone_move('forward')`
- *"Turn left"* → Should use `drone_turn('left')`  
- *"Take a picture"* → Should use `drone_take_picture()`
- *"Go back"* → Should use `drone_move('backward')`

**Alternative Phrasings:**
- *"Step ahead"*, *"Go forward one step"*
- *"Rotate right"*, *"Turn to the right"*
- *"Snap a photo"*, *"Capture what's in front"*

> **Note**: you can also try in other languages and include typos.

**What to Notice:**
- Does it choose the right tool for each command?
- Does it execute exactly ONE action?
- Can it handle different ways of saying the same thing?
- What does the LLM's reasoning look like?

In [ ]:
# Create Interactive Command Interface for your Natural Command Agent
# Make sure you've completed the agent creation above first
interactive_interface = NLGameInterface("Testing student solution", game, drone_command_agent)
interactive_interface.create_interface()

## Building a Multi-Action Command Agent (Advanced/Optional)

So far, our agent can only handle one simple command at a time. But wouldn't it be more powerful if it could understand and execute a sequence of actions from a single instruction?

For example, imagine giving commands like:

- "Move forward three times, then turn right, and take a picture."
- "Go left, move forward twice, and capture an image."
- "Turn around and go straight ahead."

This is an **advanced and optional task**. If you'd like a challenge, try enhancing the `drone_command_agent` to process these more complex, multi-step commands.

Think about how you would modify the system prompt to guide the agent to:
1. Identify multiple actions within a single command.
2. Determine the correct sequence of tools to call.
3. Handle quantities (like "three times" or "twice").
4. Execute each action in order.
5. Provide feedback on the result of each action.

Modify the `drone_command_agent` code in the cells below to enable this multi-action capability! If you prefer, you can skip this and proceed to the provided solutions.

In [ ]:
# TODO: STUDENT TASK (Advanced) - Create your Multi-Action Command Agent

# Reset the game for a fresh start
game = DroneSafariGame()

# 1. Create a new agent (reuse LLMAgent class)
multi_action_agent = LLMAgent(model=model_name)

# 2. Add your tools to the agent
# multi_action_agent.tools = [___, ...]

# 3. Write an enhanced system prompt here
# Think about how to instruct the agent to identify multiple actions,
# handle quantities (like "twice" or "three times"),
# and execute them in sequence.
multi_action_agent.system_prompt = """
Your enhanced system prompt here!
"""

In [ ]:
# 🧪 Test Your Multi-Action Command Agent - Interactive Interface
# Make sure you've completed the multi-action agent creation above first

title = "Testing Student Multi-Action Solution"
interactive_multi_action_interface = NLGameInterface(title, game, multi_action_agent)
interactive_multi_action_interface.create_interface()

## Working with Sensor Data (Advanced/Optional)

Now, let's explore how the agent can make decisions based on sensory input from the drone. By adding `sensors=True` to the game action calls (`drone_move`, `drone_turn`, `drone_take_picture`), the game will provide a summary of everything within a 2-cell radius of the drone.

This is another **advanced and optional task**. Using sensor data will allow the agent to potentially play the game more strategically, making decisions based on the environment rather than just following direct commands.

## Example: Moving with Sensors

Let's see how calling a movement function with `sensors=True` changes the output.

In [ ]:
# Reset the game for a fresh start
game = DroneSafariGame()

# Example of calling a move function with sensors=True
print("🚁 Moving forward with sensors enabled:")
result = game.move('forward', sensors=True)
print(result)

# Visualize the game state to see the updated position and potentially sensor output on the map
print("\n🗺️ Safari Map with Sensor Data:")
game.visualize(figsize=(8, 6))

# You can also try this with turn() and take_picture()
# print("\n🔄 Turning right with sensors enabled:")
# result = game.turn('right', sensors=True)
# print(result)
# game.visualize(figsize=(8, 6))

# print("\n📷 Taking picture with sensors enabled:")
# result = game.take_picture(sensors=True)
# print(result)
# game.visualize(figsize=(8, 6))

## Towards an Autonomous Agent

You've seen how adding sensor data to the tool outputs provides the drone with information about its environment. Now, imagine an agent that can not only receive this data but also use it to make its own decisions about what to do next, without explicit commands from you for each step!

Building a truly autonomous agent that can strategize and play the game on its own based on sensory input is a more complex task. It requires the agent to:

1.  **Interpret** the sensor data to understand the game state (where are the animals? where are the obstacles?).
2.  **Determine** the best course of action based on its mission (find animals, avoid crashes).
3.  **Select** the appropriate tool (move, turn, picture) and parameters.
4.  **Use its own output (the result of the action, including new sensor data) as the input for the next decision.**

This creates a feedback loop where the agent continuously observes, decides, and acts.

Provided Solution 3 demonstrates one approach to building such an autonomous agent. It shows how to modify the tools to consistently include sensor data and how to craft a system prompt that guides the agent's decision-making process based on this information and the game's objectives.

## Provided Solution #1 - One action at a time

Here's a working example of the Natural Command Agent. Study how the system prompt guides the agent's behavior!

Below you'll get another **interactive interface** to test the solution agent and see how it performs compared to your implementation.

In [ ]:
# Provided solution to the Natural Command Agent task

# Reset the game for a fresh start
game = DroneSafariGame()

# 1. Create a new agent
natural_command_agent = LLMAgent(model=model_name)

# 2. Add the tools to the agent (only the 3 action tools)
natural_command_agent.tools = [drone_move, drone_turn, drone_take_picture]

# 3. Write a simple, focused system prompt
natural_command_agent.system_prompt = """
You are a drone command translator. Your job is to translate natural language commands into single game actions.
After executing the action, summarize nicely what you did and the relevant consequences in the game.

AVAILABLE ACTIONS:
- drone_move(move_type): Move 'forward', 'backward', 'left', or 'right'
- drone_turn(turn_direction): Turn 'left' or 'right'
- drone_take_picture(): Take a photograph

COMMAND TRANSLATION:
- "move forward" / "step ahead" / "go forward" → drone_move('forward')
- "move back" / "back up" / "reverse" → drone_move('backward')
- "move left" / "step left" → drone_move('left')
- "move right" / "step right" → drone_move('right')
- "turn left" / "rotate left" → drone_turn('left')
- "turn right" / "rotate right" → drone_turn('right')
- "take picture" / "snap photo" / "capture" → drone_take_picture()

Execute exactly ONE action per command.
"""

In [ ]:
# Interactive Interface for the Solution Agent
title = "Testing Provided Solution #1: One Action per User Command"
solution_interface = NLGameInterface(title, game, natural_command_agent)
solution_interface.create_interface()

## Provided Solution 2: Multi-Action Command Agent

The first solution was great for single commands, but what if you want to give **complex multi-step instructions** in natural language?

**Examples of what you'll be able to say:**
- *"Move forward twice, turn right, and take a picture"*
- *"Go left, move forward, turn around, and snap a photo"*
- *"Navigate forward, right, then capture what's there"*
- *"Turn left, step forward three times, turn right, and photograph"*

**Key Innovation:** This agent can **parse complex commands** and **execute multiple actions in sequence** while providing detailed feedback about each step.

**What's Different:**
- 🔄 **Sequence Processing**: Breaks down complex commands into individual actions
- 📋 **Step-by-Step Execution**: Performs each action and reports the result
- 🎯 **Better Strategy Support**: You can plan multi-step moves in one command
- 🔍 **Detailed Feedback**: See exactly what happened at each step

Let's build this enhanced agent!

In [ ]:
# Enhanced Multi-Action Agent - Solution Implementation

# Reset the game for a fresh start
game = DroneSafariGame()

# Create the Multi-Action Agent
multi_action_agent = LLMAgent(model=model_name)

# Add the enhanced tools
multi_action_agent.tools = [drone_move, drone_turn, drone_take_picture]

# Write an enhanced system prompt for multi-action commands
multi_action_agent.system_prompt = """
You are an advanced drone command agent that can execute multiple actions in sequence from a single natural language command.
After executing the actions, summarize nicely what you did and the relevant consequences in the game.

AVAILABLE ACTIONS:
- drone_move(move_type): Move 'forward', 'backward', 'left', or 'right'
- drone_turn(turn_direction): Turn 'left' or 'right'
- drone_take_picture(): Take a photograph

MULTI-ACTION CAPABILITY:
You can execute MULTIPLE actions in sequence from complex commands like:
- "Move forward twice, turn right, take picture" → drone_move('forward'), drone_move('forward'), drone_turn('right'), drone_take_picture()
- "Go left, forward, turn around, snap photo" → drone_move('left'), drone_move('forward'), drone_turn('left'), drone_turn('left'), drone_take_picture()
- "Step backward, turn left, move forward three times" → drone_move('backward'), drone_turn('left'), drone_move('forward'), drone_move('forward'), drone_move('forward')

COMMAND PARSING RULES:
- "twice" or "two times" = execute action 2 times
- "three times" = execute action 3 times
- "turn around" = turn('left'), turn('left') OR turn('right'), turn('right')
- Multiple actions separated by "then", "and", commas = execute in sequence

Remember: Execute ALL actions in the sequence - don't stop after the first one!
"""

In [ ]:
# Interactive Interface for Multi-Action Agent
title = "Testing Provided Solution #2: Multiple Actions in One Command"
multi_action_interface = NLGameInterface(title, game, multi_action_agent)
multi_action_interface.create_interface()

In [ ]:
# Example Multi-Action Commands to Test
multi_action_examples = [
    "Move forward twice, turn right, and take a picture",
    "Go left, step forward, turn around, and snap a photo",
    "Navigate forward two times, rotate left, capture image",
]

# To win the game in three commands (execute in sequence)
winning_commands = [
    "Move forward 4 times, move right, forward again, right 2 times, turn around, and take a photo",
    "Move right 2 times, move forward 9 times, turn left, and take a picture",
    "Move right, move back 6 times, move left 2 times, and take a picture"
]

# Provided Solution 3: Autonomous Agent with Sensors

The previous solutions focused on translating human commands into single or multiple actions. Now, we'll explore how an agent can make decisions **autonomously** based on sensory input from the game environment.

This solution modifies the existing tools to include sensor data in their output and creates a new agent designed to interpret this data and make strategic decisions to complete the game mission.

**Key Concepts:**

*   **Sensor Data:** The game provides a summary of objects (animals, trees) within a 2-cell radius of the drone. This data is crucial for the autonomous agent to understand its surroundings.
*   **Autonomous Decision Making:** The agent will use the sensor data, combined with its mission objectives (find all animals, avoid obstacles), to determine the next best action without explicit human instruction for each step.
*   **Connecting Output to Input:** The output of each agent action (including sensor data) will be fed back as input for the agent's next decision, creating a continuous loop of observation and action.

## Modify Tools to Include Sensors

To enable the autonomous agent to receive information about the environment, we need to modify the `drone_move`, `drone_turn`, and `drone_take_picture` tools. We'll add an optional `sensors` parameter (defaulting to `False`) to each tool. When `sensors` is set to `True`, the game will return a string that includes a summary of objects within the drone's sensor range, in addition to the result of the action.

This sensor data will be the primary input for our autonomous agent's decision-making process.

In [ ]:
# Tool 1: Drone Movement with sensor information
@add_parameters_schema(
    move_type={
        "type": "string",
        "description": "Movement type: 'forward', 'backward', 'left', 'right'"
    },
    sensors={
        "type": "boolean",
        "description": "Include sensor data in the response"
    }
)
def drone_move(move_type: str, sensors: bool = False) -> str:
    """Move the drone in the specified move_type: either forward, backward, left or right"""
    result = game.move(move_type, sensors=sensors)
    if sensors and "Sensors summary:" in result:
        return result
    return result

# Tool 2: Drone Turning with sensor information
@add_parameters_schema(
    turn_direction={
        "type": "string",
        "description": "Turn direction: 'left' or 'right'"
    },
    sensors={
        "type": "boolean",
        "description": "Include sensor data in the response"
    }
)
def drone_turn(turn_direction: str, sensors: bool = False) -> str:
    """Turn the drone left or right"""
    result = game.turn(turn_direction, sensors=sensors)
    if sensors and "Sensors summary:" in result:
        return result
    return result

# Tool 3: Taking Pictures with sensor information
@add_parameters_schema(
    sensors={
        "type": "boolean",
        "description": "Include sensor data in the response"
    }
)
def drone_take_picture(sensors: bool = False) -> str:
    """Take a picture with the drone camera"""
    result = game.take_picture(sensors=sensors)
    if sensors and "Sensors summary:" in result:
        return result
    return result

## Create the Autonomous Agent

Now, let's create a new `LLMAgent` instance specifically configured for autonomous play. This agent will have access to the modified tools that provide sensor data.

The key to the autonomous agent is its **system prompt**. This prompt will guide the agent to:

*   Understand its mission (find and photograph all animals, avoid obstacles).
*   Interpret the sensor data provided after each action.
*   Develop a strategy based on the sensor data and game status.
*   Choose the next appropriate action (move, turn, or take picture) using the available tools.
*   Provide a structured output that includes a text summary, the suggested next action, and the current game status, which will be used as the input for the subsequent turn.

In [ ]:
# TODO: STUDENT TASK (Advanced/Optional) - Create the Autonomous Agent
# This agent will use sensor data to decide its actions

# Reset the game for a fresh start for the autonomous agent
game = DroneSafariGame()

# Create the autonomous agent
autonomous_agent = LLMAgent(model="gemini/gemini-2.5-flash") #gemini/gemini-2.5-flash-lite

# Assign the modified tools to the agent
autonomous_agent.tools = [drone_move, drone_turn, drone_take_picture]

# Define the system prompt for the autonomous agent
autonomous_agent.system_prompt = """
You are an **autonomous drone** navigating a safari park. Your mission is to find and photograph all three animal species (zebra, elephant, oryx) while avoiding trees and staying within the park boundaries. You can only take up to 5 pictures.
You will get a summary of the current status, latest actions, and the action or tool that should be called next.
In case that information in missing, just trigger a turn action so that we can start getting feedback from the drone sensors.

IMPORTANT: After tool calls, always provide a final answer with text summary, next action to take, and current status as it is key input for the next step.

---
### Map size:
The map is 12x12, with rows that can be visited ranging from 0 to 11.


### **Interpreting Sensor Data**

The sensor summary provides relative positions. For example, "Sensors summary: oryx GPS at (4S, 2E); zebra GPS at (3S, 4W); elephant GPS at (3N, 2E);tree at (1N, 0E)." means:
- The oryx is 4 cells South, and 2 cells East.
- The zebra is 3 cells South, and 4 cells West.
- The elephant is 3 cells North, and 0 cells East.
- A tree is 1 cell North.
- A zebra is 2 cells South and 1 cell West.

To take a picture of an animal it must be exactly 2 cells away in the facing direction. So if we are facing East, the animal should be at (0N, 2E) to be captured succesfully.

---

### **Decision-Making Guidelines**

1.  **Prioritize finding animals:** If you haven't photographed all three animal species, focus on exploring the environment.
2.  **Approach animals:** If an animal is detected and is NOT scared, move towards it. Always maintain a distance of 2 cells to avoid scaring it and ending the game.
3.  **Avoid obstacles:** If a tree or park boundary is detected in your path or nearby, change direction to avoid a collision, which will also end the game. Do not change direction until the tree or boundary is right in front of the drone.
4.  **Take pictures strategically:** Take a picture if an animal is detected exactly 2 cells away AND a tree is NOT in the middle. Do not waste pictures on animals you've already photographed or if no animal is in range.
5.  **One action type per turn:** Execute only ONE tool call (move, turn, or take_picture) based on the suggested next action. If there is no suggested action or not input information, do a turn movement to get feedback from the sensors without risking crashing into something.

---

### **Available Actions**

- `drone_move(move_type)`: `move_type` can be 'forward', 'backward', 'left', or 'right'.
- `drone_turn(turn_direction)`: `turn_direction` can be 'left' or 'right'.
- `drone_take_picture()`: Take a picture.

---

### **Example Output Format**

**Text Summary:**
I successfully took a picture of the oryx! This is great news, as it's the first of the three required animals. I now have 4 pictures remaining. My sensors still detect the oryx 2 cells to my South and a tree 1 cell to my West. Since I've already photographed the oryx, my next goal is to find the zebra and the elephant.

**Next action - A(N):**
To explore new areas and avoid the tree to my West, I will turn left.

**Current Status:**
* **Position:** (4, 9)
* **Facing:** South
* **Animals Photographed:** Oryx
* **Pictures Remaining:** 4
* **Last 5 Actions:**
A(N-1).  Took a picture of an oryx.
A(N-2).  Moved forward.
A(N-3).  Turned right.
A(N-4).  Moved forward.
A(N-5).  Turned left.
"""

## Test autonomous agent
Test the autonomous agent by running the game loop and observing its behavior.


In [ ]:
import time
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
import re # Import regex for more robust parsing
import ipywidgets as widgets # Import ipywidgets

# Reset the game state for the autonomous agent before starting the loop
game = DroneSafariGame()

# Create a button to advance the game step by step
next_step_button = widgets.Button(description="Next Step", button_style='info', layout=widgets.Layout(width='150px', margin='10px auto'))
output_area = widgets.Output()

# Set current status and sensors
current_status_and_sensors = "Game just started, you can start playing to get sensory information"

def on_next_step_button_click(b):
    """Handler for the Next Step button click."""
    global current_status_and_sensors # Declare as global to modify the variable

    # Change button to thinking state
    next_step_button.description = "Thinking..."
    next_step_button.disabled = True
    next_step_button.button_style = 'warning'

    if game.get_status()['game_over']:
        with output_area:
            clear_output(wait=True)
            print("🎮 Game Over! Click Restart to play again.")
        # Reset button state even if game is over
        next_step_button.description = "Next Step"
        next_step_button.disabled = False
        next_step_button.button_style = 'info'
        # Display the button again for restart if needed (optional)
        # display(next_step_button)
        return # Stop processing if the game is over

    with output_area:
        clear_output(wait=True)

        # Assuming the agent's chat method internally handles tool execution and
        # returns the result to be used as the next input.
        agent_response = autonomous_agent.chat(current_status_and_sensors, debug=True, clear_memory=False) # Added debug=True
        # Check if agent_response has a debug_output attribute
        if hasattr(autonomous_agent, 'debug_output') and autonomous_agent.debug_output:
                print("🔍 Agent Debug Information:")
                for debug_line in autonomous_agent.debug_output:
                    print(debug_line)

        current_status_and_sensors = agent_response # Use the agent's output as the next input

        # Display the button before the game visualization
        display(next_step_button)

        # Display the game visualization after the agent's action
        game.visualize(figsize=(6, 6))
        plt.show()
        plt.close('all') # Close the figure to prevent accumulation

        # Check if game is over after the action
        if game.get_status()['game_over']:
             print("="*50)
             print("🎮 Game Over!")
             final_status = game.get_status()
             if final_status['game_won']:
                 print("🎉 Congratulations! You won the game!")
             else:
                 print("😭 Game lost. Better luck next time!")
             print(f"Final Status: {final_status}")
             next_step_button.disabled = True # Disable the button when game is over
             # Remove the button if the game is over
             next_step_button.layout.visibility = 'hidden'
        else:
            # Reset button state after processing
            next_step_button.description = "Next Step"
            next_step_button.disabled = False
            next_step_button.button_style = 'info'
            # The button is already displayed before the visualization

# Link the button to the handler function
next_step_button.on_click(on_next_step_button_click)

# Display the output area initially
display(output_area)

# Display the initial state in the output area and then the first button
with output_area:
    clear_output(wait=True)
    print("🚀 Starting Autonomous Game Simulation...")
    print("="*50)
    # Display the button before the initial visualization
    display(next_step_button)
    game.visualize(figsize=(6, 6))
    plt.show()
    plt.close('all') # Close the figure to prevent accumulation